# arXiv HTML Reference 

 xử lý HTML cho dữ liệu arXiv:

1. **Fetch HTML** từ arXiv theo danh sách metadata có sẵn
2. **Parse HTML v2** (dual-mode strict/relaxed, tự động chọn bản tốt hơn)
3. **QA so với PyMuPDF** để phát hiện case HTML bị hụt mạnh
4. **Tính word-level Jaccard** (heavy normalize để loại nhiễu math/citation)
5. **Ratio filter** bỏ HTML hụt, xuất **near-dup JSONL** sạch để dùng trong benchmark

## Output chính
- `arxiv_html_reference_workspace_v2/parsed_jsonl/arxiv_html_reference_v2.jsonl`
- `arxiv_html_reference_workspace_v2/parsed_jsonl/arxiv_html_reference_v2_neardup.jsonl`
- `arxiv_html_reference_workspace_v2/metadata/arxiv_html_reference_v2_meta.csv`
- `arxiv_html_reference_workspace_v2/analysis/html_vs_pymupdf_jaccard.csv`
- `arxiv_html_reference_workspace_v2/analysis/html_vs_pymupdf_jaccard_summary.json`

In [31]:
%pip -q install requests beautifulsoup4 lxml tqdm pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


## 1. Import và cấu hình

In [32]:
from pathlib import Path
import re
import json
import time
import random
import hashlib
import threading
import concurrent.futures
from collections import Counter

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

random.seed(42)

# ===== Input =====
PREV_META_DIR = Path("arxiv_ocr_benchmark_workspace/metadata")
META_JSONL = PREV_META_DIR / "arxiv_search_results_multi_query.jsonl"
META_CSV   = PREV_META_DIR / "arxiv_search_results_multi_query.csv"

PYMUPDF_JSONL = Path("arxiv_ocr_benchmark_workspace/parsed_jsonl/pymupdf.jsonl")

# ===== Output =====
BASE_DIR   = Path("arxiv_html_reference_workspace_v2")
HTML_DIR   = BASE_DIR / "html_raw"
TEXT_DIR   = BASE_DIR / "html_text"
PARSED_DIR = BASE_DIR / "parsed_jsonl"
META_DIR   = BASE_DIR / "metadata"
ANALYSIS_DIR = BASE_DIR / "analysis"

for d in [HTML_DIR, TEXT_DIR, PARSED_DIR, META_DIR, ANALYSIS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ===== Fetch config =====
NUM_WORKERS = 4
SLEEP_SEC   = 1.2
MAX_RETRIES = 5
TIMEOUT     = 30
MIN_CHARS   = 300

# ===== QA gate =====
GOOD_RATIO = 0.50       # rất ổn
WARN_RATIO = 0.35       # dưới ngưỡng này coi như HTML hụt mạnh so với PyMuPDF

print(f"Metadata JSONL : {META_JSONL}  exists={META_JSONL.exists()}")
print(f"Metadata CSV   : {META_CSV}  exists={META_CSV.exists()}")
print(f"PyMuPDF JSONL  : {PYMUPDF_JSONL}  exists={PYMUPDF_JSONL.exists()}")
print(f"Output dir     : {BASE_DIR.resolve()}")

Metadata JSONL : arxiv_ocr_benchmark_workspace\metadata\arxiv_search_results_multi_query.jsonl  exists=True
Metadata CSV   : arxiv_ocr_benchmark_workspace\metadata\arxiv_search_results_multi_query.csv  exists=True
PyMuPDF JSONL  : arxiv_ocr_benchmark_workspace\parsed_jsonl\pymupdf.jsonl  exists=True
Output dir     : D:\Project\LSHBloomExperiments\notebook\arxiv_html_reference_workspace_v2


## 2. Load metadata + build HTML URL

In [33]:
if META_JSONL.exists():
    rows = []
    with open(META_JSONL, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    df = pd.DataFrame(rows)
    print(f"Loaded {len(df):,} records từ JSONL")
elif META_CSV.exists():
    df = pd.read_csv(META_CSV)
    print(f"Loaded {len(df):,} records từ CSV")
else:
    raise FileNotFoundError(
        f"Không tìm thấy metadata. Cần một trong hai file:\n  {META_JSONL}\n  {META_CSV}"
    )

def build_html_url(arxiv_id: str) -> str:
    return f"https://arxiv.org/html/{arxiv_id}"

df["html_url"] = df["arxiv_id"].astype(str).apply(build_html_url)

print(df[["arxiv_id", "title", "primary_category", "published", "html_url"]].head(5).to_string(index=False))

Loaded 1,579 records từ JSONL
    arxiv_id                                                                                                                      title primary_category            published                            html_url
2604.01212v1                                $\texttt{YC-Bench}$: Benchmarking AI Agents for Long-Term Planning and Consistent Execution            cs.CL 2026-04-01T17:52:19Z https://arxiv.org/html/2604.01212v1
2604.01195v1                                         ORBIT: Scalable and Verifiable Data Generation for Search Agents on a Tight Budget            cs.CL 2026-04-01T17:42:41Z https://arxiv.org/html/2604.01195v1
2604.01193v1                                                           Embarrassingly Simple Self-Distillation Improves Code Generation            cs.CL 2026-04-01T17:39:50Z https://arxiv.org/html/2604.01193v1
2604.01181v1 True (VIS) Lies: Analyzing How Generative AI Recognizes Intentionality, Rhetoric, and Misleadingness in Visualization

## 3. Load PyMuPDF text map để QA đối chiếu

In [34]:
pymupdf_map = {}

if not PYMUPDF_JSONL.exists():
    raise FileNotFoundError(f"Không tìm thấy {PYMUPDF_JSONL}")

with open(PYMUPDF_JSONL, encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        d = json.loads(line)
        doc_id = str(d.get("doc_id", ""))
        text = d.get("text") or ""
        if doc_id and text:
            pymupdf_map[doc_id] = text

print(f"Loaded {len(pymupdf_map):,} PyMuPDF records")
sample_doc = next(iter(pymupdf_map))
print("Sample doc_id:", sample_doc, "| chars:", len(pymupdf_map[sample_doc]))

Loaded 1,578 PyMuPDF records
Sample doc_id: 2507.21563v3 | chars: 66139


## 4. Fetcher + parser HTML v2

In [35]:
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; arxiv-html-crawler/2.0; research-only)",
    "Accept": "text/html,application/xhtml+xml",
    "Accept-Language": "en-US,en;q=0.9",
})

MAX_HTML_BYTES = 30 * 1024 * 1024  # 30 MB cap per response

def fetch_html(url: str, timeout: int = TIMEOUT, max_retries: int = MAX_RETRIES):
    last_status = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = SESSION.get(url, timeout=timeout, allow_redirects=True, stream=True)
            last_status = resp.status_code
            if resp.status_code == 200:
                chunks = []
                total = 0
                for chunk in resp.iter_content(chunk_size=64 * 1024):
                    if chunk:
                        total += len(chunk)
                        if total > MAX_HTML_BYTES:
                            print(f"[SizeLimit] {url[:90]} exceeds {MAX_HTML_BYTES//1024//1024} MB, truncating")
                            chunks.append(chunk[: MAX_HTML_BYTES - (total - len(chunk))])
                            break
                        chunks.append(chunk)
                raw_bytes = b"".join(chunks)
                encoding = resp.encoding or "utf-8"
                try:
                    html_text = raw_bytes.decode(encoding, errors="replace")
                except (LookupError, TypeError):
                    html_text = raw_bytes.decode("utf-8", errors="replace")
                return html_text, 200
            if resp.status_code == 404:
                return None, 404
            if resp.status_code in (429, 500, 502, 503, 504):
                wait = (2 ** attempt) + random.uniform(0, 1.0)
                print(f"[HTTP {resp.status_code}] {url[:90]} -> wait {wait:.1f}s ({attempt}/{max_retries})")
                time.sleep(wait)
                continue
            return None, resp.status_code
        except MemoryError:
            print(f"[MemoryError] {url[:90]} - skipping")
            return None, "memory_error"
        except requests.exceptions.Timeout:
            wait = (2 ** attempt) + random.uniform(0, 1.0)
            print(f"[Timeout] {url[:90]} -> wait {wait:.1f}s ({attempt}/{max_retries})")
            time.sleep(wait)
        except requests.exceptions.RequestException as e:
            wait = (2 ** attempt) + random.uniform(0, 1.0)
            print(f"[Error] {e} -> wait {wait:.1f}s ({attempt}/{max_retries})")
            time.sleep(wait)
    return None, last_status

REMOVE_TAGS_HARD = [
    "script", "style", "noscript", "nav", "footer", "header",
    "button", "form", "input", "select", "textarea"
]

MAIN_SELECTORS_STRICT = [
    "article",
    "section.ltx_document",
    "div.ltx_document",
    "div.ltx_page_main",
    "main",
    "div#content",
]

MAIN_SELECTORS_RELAXED = MAIN_SELECTORS_STRICT + [
    "div.ltx_page_content",
    "body",
]

DROP_SELECTORS_STRICT = [
    ".ltx_bibliography",
    ".ltx_page_footer",
    ".ltx_page_header",
    ".ltx_toclist",
    ".ltx_role_footnote",
    ".ltx_note_mark",
    ".ltx_dates",
    ".ltx_authors",
]

DROP_SELECTORS_RELAXED = [
    ".ltx_page_footer",
    ".ltx_page_header",
    ".ltx_toclist",
    ".ltx_dates",
]

def sha1_text(text: str) -> str:
    return hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()

def _clean_lines(text: str) -> str:
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    lines = []
    for line in text.split("\n"):
        s = line.strip()
        if not s:
            continue
        if len(s) <= 2 and not any(ch.isalpha() for ch in s):
            continue
        if re.fullmatch(r"[\[\]\(\)\{\}\d,.;:*\-–—]+", s):
            continue
        lines.append(s)
    return "\n".join(lines).strip()

def _pick_main_node(soup: BeautifulSoup, selectors):
    candidates = []
    for sel in selectors:
        for node in soup.select(sel):
            raw_len = len(node.get_text(" ", strip=True))
            if raw_len > 1000:
                candidates.append((raw_len, sel, node))
    if not candidates:
        return None, None
    candidates.sort(key=lambda x: x[0], reverse=True)
    raw_len, sel, node = candidates[0]
    return node, sel

def parse_html_to_text_v2(
    html: str,
    selectors,
    drop_selectors,
    keep_math_alt: bool = True,
    keep_figcaptions: bool = True,
):
    soup = BeautifulSoup(html, "lxml")

    for tag_name in REMOVE_TAGS_HARD:
        for t in soup.find_all(tag_name):
            t.decompose()

    main, chosen_selector = _pick_main_node(soup, selectors)
    if main is None:
        return "", {"chosen_selector": None}

    for sel in drop_selectors:
        for t in main.select(sel):
            t.decompose()

    if keep_figcaptions:
        for fig in main.find_all("figure"):
            fig.unwrap()
    else:
        for t in main.find_all(["figure", "figcaption"]):
            t.decompose()

    if keep_math_alt:
        for m in main.find_all("math"):
            alt = m.get("alttext") or m.get("aria-label") or m.get_text(" ", strip=True)
            alt = (alt or "").strip()
            if alt:
                m.replace_with(" " + alt + " ")
            else:
                m.decompose()
    else:
        for m in main.find_all("math"):
            m.decompose()

    text = main.get_text(separator="\n")
    text = _clean_lines(text)
    return text, {"chosen_selector": chosen_selector}

def choose_best_html_text(html_raw: str, pymupdf_text: str | None):
    pdf_len = len(pymupdf_text or "")

    text1, meta1 = parse_html_to_text_v2(
        html_raw,
        selectors=MAIN_SELECTORS_STRICT,
        drop_selectors=DROP_SELECTORS_STRICT,
        keep_math_alt=True,
        keep_figcaptions=True,
    )
    ratio1 = (len(text1) / pdf_len) if pdf_len else None

    text2, meta2 = parse_html_to_text_v2(
        html_raw,
        selectors=MAIN_SELECTORS_RELAXED,
        drop_selectors=DROP_SELECTORS_RELAXED,
        keep_math_alt=True,
        keep_figcaptions=True,
    )
    ratio2 = (len(text2) / pdf_len) if pdf_len else None

    candidates = [
        ("strict", text1, meta1, ratio1),
        ("relaxed", text2, meta2, ratio2),
    ]

    if pdf_len:
        best_name, best_text, best_meta, best_ratio = max(
            candidates,
            key=lambda x: (-1 if x[3] is None else x[3], len(x[1]))
        )
    else:
        best_name, best_text, best_meta, best_ratio = max(
            candidates,
            key=lambda x: len(x[1])
        )

    html_len = len(best_text)

    if pdf_len == 0:
        qa_status = "no_pdf_reference"
    elif best_ratio is None:
        qa_status = "no_pdf_reference"
    elif best_ratio >= GOOD_RATIO:
        qa_status = "ok"
    elif best_ratio >= WARN_RATIO:
        qa_status = "warn_low_coverage_vs_pdf"
    else:
        qa_status = "low_coverage_vs_pdf"

    qa = {
        "qa_status": qa_status,
        "html_chars": html_len,
        "pdf_chars": pdf_len,
        "char_ratio": None if best_ratio is None else round(best_ratio, 4),
        "chosen_parse_mode": best_name,
        "chosen_selector": best_meta.get("chosen_selector"),
        "strict_chars": len(text1),
        "relaxed_chars": len(text2),
    }
    return best_text, qa

print("Fetcher + parser v2 ready.")


Fetcher + parser v2 ready.


## 5. Test nhanh 1 bài

In [36]:
sample = df.iloc[0]
print(f"Doc  : {sample['arxiv_id']}")
print(f"Title: {str(sample['title'])[:100]}")
print(f"URL  : {sample['html_url']}")

html_raw, status = fetch_html(sample["html_url"])
if html_raw:
    text, qa = choose_best_html_text(html_raw, pymupdf_map.get(str(sample["arxiv_id"]), ""))
    print(f"HTTP={status} | text_chars={len(text):,}")
    print("QA:", qa)
    print("\n--- Preview (800 chars) ---\n")
    print(text[:800])
else:
    print(f"Fetch failed: HTTP={status}")

Doc  : 2604.01212v1
Title: $\texttt{YC-Bench}$: Benchmarking AI Agents for Long-Term Planning and Consistent Execution
URL  : https://arxiv.org/html/2604.01212v1
HTTP=200 | text_chars=40,909
QA: {'qa_status': 'ok', 'html_chars': 40909, 'pdf_chars': 51846, 'char_ratio': 0.789, 'chosen_parse_mode': 'relaxed', 'chosen_selector': 'body', 'strict_chars': 40806, 'relaxed_chars': 40909}

--- Preview (800 chars) ---

License: CC BY 4.0
arXiv:2604.01212v1 [cs.CL] 01 Apr 2026
\ycbench
: Benchmarking AI Agents for Long-Term Planning and Consistent Execution
Muyu He
Adit Jain
Anand Kumar
Vincent Tu
Soumyadeep Bakshi
Sachin Patro
Nazneen Rajani
Abstract
As LLM agents tackle increasingly complex tasks, a critical question is whether they can maintain strategic coherence over long horizons: planning under uncertainty, learning from delayed feedback, and adapting when early mistakes compound. We introduce
\ycbench
, a benchmark that evaluates these capabilities by tasking an agent with running a simul

## 6. Fetch + parse toàn bộ, có resume

In [37]:
RESUME_PATH = PARSED_DIR / "arxiv_html_reference_v2.jsonl"
_lock = threading.Lock()

done_ids = set()
valid_records = []

if RESUME_PATH.exists():
    with open(RESUME_PATH, encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
                done_ids.add(rec["doc_id"])
                valid_records.append(rec)
            except Exception:
                pass

    with open(RESUME_PATH, "w", encoding="utf-8") as f:
        for rec in valid_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print(f"Resume loaded: {len(done_ids):,} records")

records = list(valid_records)
errors = []

todo_df = df[~df["arxiv_id"].astype(str).isin(done_ids)].copy()
print(f"Cần fetch: {len(todo_df):,} | Đã có: {len(done_ids):,}")

def process_one(row: dict):
    arxiv_id = str(row.get("arxiv_id", ""))
    html_url = str(row.get("html_url", ""))
    title    = str(row.get("title", ""))
    cat      = str(row.get("primary_category", ""))
    pub      = str(row.get("published", ""))
    pdf_url  = str(row.get("pdf_url", ""))

    safe_id   = re.sub(r"[^\w.\-]", "_", arxiv_id)[:120]
    html_path = HTML_DIR / f"{safe_id}.html"
    text_path = TEXT_DIR / f"{safe_id}.txt"

    try:
        html_raw, status = fetch_html(html_url)
    except MemoryError:
        return None, {"doc_id": arxiv_id, "error": "MemoryError during fetch"}

    time.sleep(SLEEP_SEC + random.uniform(0, 0.4))

    if html_raw is None:
        rec = {
            "doc_id": arxiv_id,
            "parser_name": "arxiv_html",
            "source_type": "html_reference",
            "title": title,
            "primary_category": cat,
            "published": pub,
            "html_url": html_url,
            "pdf_url": pdf_url,
            "html_path": "",
            "text_path": "",
            "char_count": 0,
            "line_count": 0,
            "text_sha1": "",
            "text": "",
            "fetch_status": f"failed_{status}",
            "metadata": {
                "source": "arxiv",
                "http_status": status,
                "html_available": False,
            },
        }
        return rec, None

    try:
        html_path.write_text(html_raw, encoding="utf-8", errors="replace")
    except MemoryError:
        return None, {"doc_id": arxiv_id, "error": "MemoryError writing html_raw"}

    try:
        pdf_text = pymupdf_map.get(arxiv_id, "")
        text, qa = choose_best_html_text(html_raw, pdf_text)
    except MemoryError:
        return None, {"doc_id": arxiv_id, "error": "MemoryError during parse"}
    except Exception as e:
        return None, {"doc_id": arxiv_id, "error": f"parse error: {e}"}

    text_path.write_text(text, encoding="utf-8", errors="replace")

    status_ok = len(text) >= MIN_CHARS and qa["qa_status"] in {"ok", "warn_low_coverage_vs_pdf", "no_pdf_reference"}
    fetch_status = "ok" if status_ok else qa["qa_status"]

    rec = {
        "doc_id": arxiv_id,
        "parser_name": "arxiv_html",
        "source_type": "html_reference",
        "title": title,
        "primary_category": cat,
        "published": pub,
        "html_url": html_url,
        "pdf_url": pdf_url,
        "html_path": str(html_path),
        "text_path": str(text_path),
        "char_count": len(text),
        "line_count": text.count("\n") + 1 if text else 0,
        "text_sha1": sha1_text(text) if text else "",
        "text": text,
        "fetch_status": fetch_status,
        "metadata": {
            "source": "arxiv",
            "http_status": 200,
            "html_available": True,
            "summary": str(row.get("summary", "")),
            "authors": row.get("authors", []),
            "categories": row.get("categories", []),
            **qa,
        },
    }
    return rec, None

todo_rows = todo_df.to_dict(orient="records")
newly_done = 0

out_f = open(RESUME_PATH, "a", encoding="utf-8", errors="replace")
pbar = tqdm(total=len(todo_rows), desc="Fetch HTML v2")

with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    future_map = {executor.submit(process_one, row): row for row in todo_rows}
    for future in concurrent.futures.as_completed(future_map):
        try:
            rec, err = future.result()
        except MemoryError as e:
            row = future_map[future]
            err = {"doc_id": row.get("arxiv_id", "?"), "error": f"MemoryError: {e}"}
            rec = None
        with _lock:
            if rec:
                records.append(rec)
                newly_done += 1
                out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                out_f.flush()
            if err:
                errors.append(err)
        pbar.update(1)

pbar.close()
out_f.close()

status_counts = Counter(r["fetch_status"] for r in records)
print("\n=== Fetch summary ===")
print(f"Total records : {len(records):,}")
for k, v in status_counts.most_common():
    print(f"{k:28s} {v:>6,}")
print(f"Parse errors  : {len(errors):,}")
if errors:
    print("First error:", errors[0])


Resume loaded: 1,579 records
Cần fetch: 0 | Đã có: 1,579


Fetch HTML v2: 0it [00:00, ?it/s]


=== Fetch summary ===
Total records : 1,579
ok                            1,418
failed_404                      157
low_coverage_vs_pdf               4
Parse errors  : 0


## 7. Thống kê nhanh coverage / QA

In [38]:
stats_df = pd.DataFrame([
    {
        "doc_id": r["doc_id"],
        "primary_category": r["primary_category"],
        "published": str(r["published"])[:10] if r.get("published") else "",
        "char_count": r["char_count"],
        "line_count": r["line_count"],
        "fetch_status": r["fetch_status"],
        "char_ratio": (r.get("metadata") or {}).get("char_ratio"),
        "qa_status": (r.get("metadata") or {}).get("qa_status"),
    }
    for r in records
])

print("fetch_status:")
print(stats_df["fetch_status"].value_counts().to_string())

ok_df = stats_df[stats_df["fetch_status"] == "ok"]
if len(ok_df):
    print("\nchar_count describe (OK only):")
    print(ok_df["char_count"].describe().to_string())

if "char_ratio" in ok_df.columns and ok_df["char_ratio"].notna().any():
    print("\nchar_ratio describe (OK only):")
    print(ok_df["char_ratio"].dropna().describe().to_string())

display(ok_df.head(10))

fetch_status:
fetch_status
ok                     1418
failed_404              157
low_coverage_vs_pdf       4

char_count describe (OK only):
count      1418.000000
mean      60588.170663
std       38079.804441
min        8141.000000
25%       39646.000000
50%       53338.500000
75%       72004.750000
max      671448.000000

char_ratio describe (OK only):
count    1418.000000
mean        0.952580
std         0.116311
min         0.365100
25%         0.916400
50%         0.963200
75%         0.996775
max         1.766100


,doc_id,primary_category,published,char_count,line_count,fetch_status,char_ratio,qa_status
1,2604.01212v1,cs.CL,2026-04-01,40909,1111,ok,0.7890,ok
2,2604.01181v1,cs.HC,2026-04-01,77792,1780,ok,0.6138,ok
3,2604.00986v1,cs.CR,2026-04-01,40785,805,ok,0.9240,ok
4,2604.01029v1,cs.SE,2026-04-01,41981,1053,ok,0.8409,ok
5,2604.01193v1,cs.CL,2026-04-01,128821,2266,ok,0.9453,ok
6,2604.00890v1,cs.AI,2026-04-01,49947,1368,ok,0.9413,ok
7,2604.00778v1,cs.CL,2026-04-01,76395,1205,ok,0.7828,ok
8,2604.00913v1,cs.CV,2026-04-01,64335,1127,ok,0.9564,ok
9,2604.00698v1,cs.LG,2026-04-01,46550,1020,ok,0.9849,ok
10,2604.00438v1,cs.CL,2026-04-01,48717,986,ok,0.9712,ok


## 8. Xuất metadata CSV / JSONL

In [39]:
META_COLS = [
    "doc_id", "parser_name", "source_type", "title",
    "primary_category", "published", "html_url", "pdf_url",
    "html_path", "text_path", "char_count", "line_count",
    "text_sha1", "fetch_status",
]

meta_rows = [{k: r.get(k, "") for k in META_COLS} for r in records]
meta_df = pd.DataFrame(meta_rows)

csv_path   = META_DIR / "arxiv_html_reference_v2_meta.csv"
jsonl_path = META_DIR / "arxiv_html_reference_v2_meta.jsonl"

meta_df.to_csv(csv_path, index=False, encoding="utf-8")
with open(jsonl_path, "w", encoding="utf-8") as f:
    for row in meta_df.to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("CSV  :", csv_path)
print("JSONL:", jsonl_path)
print("Full text JSONL:", RESUME_PATH)

CSV  : arxiv_html_reference_workspace_v2\metadata\arxiv_html_reference_v2_meta.csv
JSONL: arxiv_html_reference_workspace_v2\metadata\arxiv_html_reference_v2_meta.jsonl
Full text JSONL: arxiv_html_reference_workspace_v2\parsed_jsonl\arxiv_html_reference_v2.jsonl


## 9. Jaccard: `arxiv_html` vs `pymupdf`

In [40]:
import unicodedata

# Regex nhận diện token là math/LaTeX artifact:
# - Chứa \, ^, {, } → LaTeX command
# - Dạng số/ký hiệu thuần: (1), (+2.3), [abc], &token
_MATH_ARTIFACT_RE = re.compile(r"[\\^{}]|^\([+-]?\d")
_PUNCT_ONLY_RE    = re.compile(r"^[\W_]+$")  # toàn dấu câu / underscore

def normalize_for_jaccard(text: str) -> set:
    """
    Normalize cho Jaccard:
    - Lowercase + NFKC unicode
    - Nối hyphen cuối dòng (PyMuPDF artifact)
    - Xóa dấu câu standalone và token LaTeX/math
    - Giữ: từ thường, số thực (0.5, 1.3), từ ghép (self-attention)
    """
    text = unicodedata.normalize("NFKC", text.lower())
    text = re.sub(r"-\s*\n\s*", "", text)           # nối hyphen cuối dòng
    text = re.sub(r"[^\w\s.\-]", " ", text)         # xóa dấu câu trừ . và -

    result = set()
    for tok in text.split():
        if len(tok) < 2:
            continue
        # Bỏ token toàn dấu câu / underscores
        if _PUNCT_ONLY_RE.match(tok):
            continue
        # Bỏ token chứa LaTeX artifact
        if _MATH_ARTIFACT_RE.search(tok):
            continue
        # Bỏ token trông như số inline dạng "0.001" đơn độc (giữ nếu là từ ghép với chữ)
        # VD: "0.5delta" vẫn giữ, "0.001" bỏ
        if re.fullmatch(r"\d+(\.\d+)?", tok):
            continue
        result.add(tok)
    return result

def word_jaccard(ta: str, tb: str) -> float:
    sa = normalize_for_jaccard(ta or "")
    sb = normalize_for_jaccard(tb or "")
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)

html_map = {}
for r in records:
    if r.get("fetch_status") == "ok" and r.get("text"):
        html_map[str(r["doc_id"])] = r["text"]

common_ids = sorted(set(html_map.keys()) & set(pymupdf_map.keys()))
print(f"HTML ok        : {len(html_map):,}")
print(f"PyMuPDF ok     : {len(pymupdf_map):,}")
print(f"Common valid   : {len(common_ids):,}")


HTML ok        : 1,418
PyMuPDF ok     : 1,578
Common valid   : 1,418


In [41]:
# ===== DEBUG: so sánh normalize strategies =====

def normalize_heavy(text: str) -> set:
    """Bản heavy gốc"""
    text = unicodedata.normalize("NFKC", text.lower())
    text = re.sub(r"-\s*\n\s*", "", text)
    text = re.sub(r"[^\w\s]", " ", text)
    result = set()
    for tok in text.split():
        if len(tok) < 2: continue
        if tok.isdigit(): continue
        if sum(c in "_{}\\^" for c in tok) >= 2: continue
        result.add(tok)
    return result

def normalize_current(text: str) -> set:
    """Bản hiện tại (calibrated)"""
    return normalize_for_jaccard(text)

def debug_normalize(doc_id: str):
    html_t = html_map.get(doc_id, "")
    pdf_t  = pymupdf_map.get(doc_id, "")
    if not html_t or not pdf_t:
        print(f"  [skip] Không có text cho {doc_id}")
        return

    html_raw_tok = set(html_t.lower().split())
    pdf_raw_tok  = set(pdf_t.lower().split())
    raw_jac      = len(html_raw_tok & pdf_raw_tok) / len(html_raw_tok | pdf_raw_tok)

    html_h = normalize_heavy(html_t)
    pdf_h  = normalize_heavy(pdf_t)
    jac_h  = len(html_h & pdf_h) / len(html_h | pdf_h)

    html_c = normalize_current(html_t)
    pdf_c  = normalize_current(pdf_t)
    jac_c  = len(html_c & pdf_c) / len(html_c | pdf_c)

    print(f"doc_id: {doc_id}  (chars html={len(html_t):,} pdf={len(pdf_t):,} ratio={len(html_t)/max(len(pdf_t),1):.3f})")
    print(f"  tokens: raw html={len(html_raw_tok):,} pdf={len(pdf_raw_tok):,}")
    print(f"  tokens: heavy html={len(html_h):,} pdf={len(pdf_h):,}   (drop {100*(1-len(html_h)/max(len(html_raw_tok),1)):.0f}% / {100*(1-len(pdf_h)/max(len(pdf_raw_tok),1)):.0f}%)")
    print(f"  tokens: current html={len(html_c):,} pdf={len(pdf_c):,}   (drop {100*(1-len(html_c)/max(len(html_raw_tok),1)):.0f}% / {100*(1-len(pdf_c)/max(len(pdf_raw_tok),1)):.0f}%)")
    print(f"  Jaccard raw     : {raw_jac:.4f}")
    print(f"  Jaccard heavy   : {jac_h:.4f}  (Δ={jac_h-raw_jac:+.4f})")
    print(f"  Jaccard current : {jac_c:.4f}  (Δ={jac_c-raw_jac:+.4f})")

    # Token giữ lại bởi current nhưng bị heavy xóa (xem có ý nghĩa không)
    kept_by_current_not_heavy = (html_c - html_h)
    alpha_only = sorted(t for t in kept_by_current_not_heavy if re.search(r"[a-z]", t))[:15]
    print(f"  Token giữ thêm so với heavy (có chữ cái): {alpha_only}")
    print()

try:
    debug_ids = jaccard_df["doc_id"].head(5).tolist()
except NameError:
    debug_ids = list(common_ids)[:5]

for did in debug_ids:
    debug_normalize(did)
    print("-" * 70)


doc_id: 2604.00963v1  (chars html=206,739 pdf=152,590 ratio=1.355)
  tokens: raw html=4,053 pdf=4,625
  tokens: heavy html=1,542 pdf=1,852   (drop 62% / 60%)
  tokens: current html=1,804 pdf=2,256   (drop 55% / 51%)
  Jaccard raw     : 0.2715
  Jaccard heavy   : 0.6962  (Δ=+0.4247)
  Jaccard current : 0.6273  (Δ=+0.3558)
  Token giữ thêm so với heavy (có chữ cái): ['-2c_', '-b', '-bounded', '-byn', '-c', '-c_', '-d', '-f_', '-ferromagnetic', '-hard', '-n', '-norm', '-o', '-o_', '-pr']

----------------------------------------------------------------------
doc_id: 2603.28046v1  (chars html=168,380 pdf=221,613 ratio=0.760)
  tokens: raw html=4,104 pdf=10,094
  tokens: heavy html=2,352 pdf=2,936   (drop 43% / 71%)
  tokens: current html=2,985 pdf=3,738   (drop 27% / 63%)
  Jaccard raw     : 0.2736
  Jaccard heavy   : 0.6311  (Δ=+0.3575)
  Jaccard current : 0.6271  (Δ=+0.3535)
  Token giữ thêm so với heavy (có chữ cái): ['-0.7x_', '-00931x_', '-0198x_', '-0215x_', '-02x_', '-03762x_', '-05

In [42]:
# ===== INSPECT chi tiết 1 doc: HTML vs PyMuPDF =====

def inspect_doc(doc_id: str, head_chars: int = 2000):
    html_t = html_map.get(doc_id, "")
    pdf_t  = pymupdf_map.get(doc_id, "")

    html_tok = normalize_for_jaccard(html_t)
    pdf_tok  = normalize_for_jaccard(pdf_t)

    only_pdf   = sorted(pdf_tok - html_tok)   # PDF có, HTML không có
    only_html  = sorted(html_tok - pdf_tok)   # HTML có, PDF không có
    common     = html_tok & pdf_tok

    print(f"{'='*80}")
    print(f"doc_id : {doc_id}")
    print(f"chars  : html={len(html_t):,}  pdf={len(pdf_t):,}  ratio={len(html_t)/max(len(pdf_t),1):.3f}")
    print(f"tokens : html={len(html_tok):,}  pdf={len(pdf_tok):,}  common={len(common):,}")
    print(f"Jaccard: {len(common)/max(len(html_tok|pdf_tok),1):.4f}")
    print()

    print(f"--- Token có trong PDF nhưng THIẾU trong HTML ({len(only_pdf):,} tokens) ---")
    # Lọc ra các từ thực sự (>= 4 chars, có chữ cái) → gợi ý nội dung bị miss
    meaningful_only_pdf = [t for t in only_pdf if len(t) >= 4 and re.search(r"[a-z]", t)]
    print("Ví dụ:", meaningful_only_pdf[:40])
    print()

    print(f"--- Token có trong HTML nhưng THỪA so với PDF ({len(only_html):,} tokens) ---")
    meaningful_only_html = [t for t in only_html if len(t) >= 4 and re.search(r"[a-z]", t)]
    print("Ví dụ:", meaningful_only_html[:40])
    print()

    print(f"--- HTML text (first {head_chars} chars) ---")
    print(html_t[:head_chars])
    print()
    print(f"--- PyMuPDF text (first {head_chars} chars) ---")
    print(pdf_t[:head_chars])
    print()

    # Tìm đoạn text dài có trong PDF mà không có trong HTML → section bị miss
    pdf_lines = [l.strip() for l in pdf_t.split("\n") if len(l.strip()) > 60]
    html_content = html_t.lower()
    missing_lines = []
    for line in pdf_lines:
        # Lấy 6 từ đầu của dòng làm probe
        probe = " ".join(line.lower().split()[:6])
        if probe and probe not in html_content:
            missing_lines.append(line)
    print(f"--- Dòng dài trong PDF không thấy trong HTML ({len(missing_lines):,} dòng) ---")
    for line in missing_lines[:15]:
        print(" ✗", line[:120])

inspect_doc("2601.05038v1")


doc_id : 2601.05038v1
chars  : html=21,166  pdf=51,658  ratio=0.410
tokens : html=1,005  pdf=2,173  common=921
Jaccard: 0.4081

--- Token có trong PDF nhưng THIẾU trong HTML (1,252 tokens) ---
Ví dụ: ['12th', '16th', '1https', '2.0e-4', '2.0e-5', '200k', '2023a', '2023a.', '2023b', '2023b.', '2025a', '2025a.', '2025b', '2025b.', '2025c', '2025c.', '2601.05038v1', '28th', '4-way', '55th', '59th', '61st', '63rd', 'aaron', 'ablation', 'about', 'absolute', 'acc.', 'accelerated', 'according', 'accumulation', 'accuracy', 'achiam', 'achieves', 'activate', 'adamw', 'adapted', 'addition', 'additionally', 'addresses']

--- Token có trong HTML nhưng THỪA so với PDF (84 tokens) ---
Ví dụ: ['achiam2023gpt', 'asai2024self', 'available', 'begin', 'bengio2013estimating', 'cdot', 'chen2025dast', 'cheng2024xrag', 'daptive', 'delta', 'deng2025silver', 'displaystyle', 'dong2024internlm', 'dots', 'ecursive', 'end-to-end', 'fusion-in-decoder', 'gecontext', 'gray', 'guu2020retrieval', 'houlsby2019parameter',

In [43]:
RATIO_MIN_NEARDUP = 0.50  # Ngưỡng lọc HTML hụt

# ── Bước 1: tính char_ratio nhanh cho tất cả common_ids ──
ratio_rows = []
for doc_id in common_ids:
    html_chars = len(html_map[doc_id])
    pdf_chars  = len(pymupdf_map[doc_id])
    ratio_rows.append({
        "doc_id": doc_id,
        "html_chars": html_chars,
        "pymupdf_chars": pdf_chars,
        "char_ratio_html_over_pdf": html_chars / max(pdf_chars, 1),
    })

ratio_df    = pd.DataFrame(ratio_rows)
neardup_ids = ratio_df[ratio_df["char_ratio_html_over_pdf"] >= RATIO_MIN_NEARDUP]["doc_id"].tolist()
dropped_ids = ratio_df[ratio_df["char_ratio_html_over_pdf"] <  RATIO_MIN_NEARDUP]["doc_id"].tolist()

print(f"Tổng common docs       : {len(common_ids):,}")
print(f"  GIỮ  (ratio >= {RATIO_MIN_NEARDUP}) : {len(neardup_ids):,}")
print(f"  LOẠI (HTML hụt)       : {len(dropped_ids):,}")
print()

# ── Bước 2: tính Jaccard chỉ trên tập đã lọc ──
rows = []
for doc_id in tqdm(neardup_ids, desc="Compute Jaccard (ratio-filtered)"):
    html_text  = html_map[doc_id]
    pdf_text   = pymupdf_map[doc_id]
    html_chars = len(html_text)
    pdf_chars  = len(pdf_text)
    char_ratio = html_chars / max(pdf_chars, 1)
    jac        = word_jaccard(html_text, pdf_text)

    rows.append({
        "doc_id": doc_id,
        "jaccard_word_unigram": jac,
        "html_chars": html_chars,
        "pymupdf_chars": pdf_chars,
        "char_ratio_html_over_pdf": char_ratio,
    })

jaccard_df = pd.DataFrame(rows).sort_values("jaccard_word_unigram").reset_index(drop=True)

summary = {
    "n_html_ok": int(len(html_map)),
    "n_pymupdf_ok": int(len(pymupdf_map)),
    "n_common_valid": int(len(common_ids)),
    "n_after_ratio_filter": int(len(neardup_ids)),
    "n_dropped_low_ratio": int(len(dropped_ids)),
    "ratio_min": RATIO_MIN_NEARDUP,
    "mean_jaccard": float(jaccard_df["jaccard_word_unigram"].mean())         if len(jaccard_df) else None,
    "median_jaccard": float(jaccard_df["jaccard_word_unigram"].median())     if len(jaccard_df) else None,
    "p10_jaccard": float(jaccard_df["jaccard_word_unigram"].quantile(0.10))  if len(jaccard_df) else None,
    "p90_jaccard": float(jaccard_df["jaccard_word_unigram"].quantile(0.90))  if len(jaccard_df) else None,
}

print("Jaccard distribution (sau khi lọc HTML hụt):")
display(jaccard_df["jaccard_word_unigram"].describe().to_frame().T)
display(jaccard_df.head(10))
print("\nSummary:")
print(json.dumps(summary, indent=2, ensure_ascii=False))


Tổng common docs       : 1,418
  GIỮ  (ratio >= 0.5) : 1,413
  LOẠI (HTML hụt)       : 5



Compute Jaccard (ratio-filtered):   0%|          | 0/1413 [00:00<?, ?it/s]

Compute Jaccard (ratio-filtered): 100%|██████████| 1413/1413 [00:39<00:00, 35.34it/s]

Jaccard distribution (sau khi lọc HTML hụt):


,count,mean,std,min,25%,50%,75%,max
jaccard_word_unigram,1413.0,0.816296,0.09723,0.363293,0.752145,0.828467,0.894929,0.995986


,doc_id,jaccard_word_unigram,html_chars,pymupdf_chars,char_ratio_html_over_pdf
0,2601.03474v2,0.363293,85492,48408,1.766072
1,2601.03262v1,0.456649,43052,82674,0.520744
2,2601.18886v1,0.467961,16749,30849,0.542935
3,2508.20408v1,0.468859,19707,31894,0.617891
4,2603.27055v1,0.476415,49473,97241,0.508767
5,2604.00069v1,0.490014,124971,200288,0.623957
6,2603.21172v1,0.491711,44820,41747,1.073610
7,2603.28589v1,0.495798,58415,99282,0.588375
8,2603.19809v1,0.496650,42486,58656,0.724325
9,2603.25189v1,0.513640,40024,51482,0.777437



Summary:
{
  "n_html_ok": 1418,
  "n_pymupdf_ok": 1578,
  "n_common_valid": 1418,
  "n_after_ratio_filter": 1413,
  "n_dropped_low_ratio": 5,
  "ratio_min": 0.5,
  "mean_jaccard": 0.8162956582011195,
  "median_jaccard": 0.8284671532846716,
  "p10_jaccard": 0.690597107236931,
  "p90_jaccard": 0.9324151067116401
}


## 10. Inspect outliers

In [44]:
print("=== Lowest Jaccard cases ===")
display(jaccard_df.head(10))

print("\n=== Highest Jaccard cases ===")
display(jaccard_df.tail(10).sort_values("jaccard_word_unigram", ascending=False))

=== Lowest Jaccard cases ===


,doc_id,jaccard_word_unigram,html_chars,pymupdf_chars,char_ratio_html_over_pdf
0,2601.03474v2,0.363293,85492,48408,1.766072
1,2601.03262v1,0.456649,43052,82674,0.520744
2,2601.18886v1,0.467961,16749,30849,0.542935
3,2508.20408v1,0.468859,19707,31894,0.617891
4,2603.27055v1,0.476415,49473,97241,0.508767
5,2604.00069v1,0.490014,124971,200288,0.623957
6,2603.21172v1,0.491711,44820,41747,1.073610
7,2603.28589v1,0.495798,58415,99282,0.588375
8,2603.19809v1,0.496650,42486,58656,0.724325
9,2603.25189v1,0.513640,40024,51482,0.777437



=== Highest Jaccard cases ===


,doc_id,jaccard_word_unigram,html_chars,pymupdf_chars,char_ratio_html_over_pdf
1412,2603.14170v1,0.995986,63864,64407,0.991569
1411,2603.14629v1,0.987867,19451,19595,0.992651
1410,2603.29075v1,0.979879,62043,62354,0.995012
1409,2603.28924v1,0.975242,46078,46255,0.996173
1408,2603.27435v1,0.971429,153673,159132,0.965695
1407,2603.27626v1,0.970963,79775,80016,0.996988
1406,2603.24535v1,0.970925,25475,26089,0.976465
1405,2603.14173v1,0.969989,43640,44300,0.985102
1404,2511.01386v1,0.968435,145667,148004,0.984210
1403,2603.26742v1,0.968429,30511,34193,0.892317


In [45]:
LOW_RATIO_THRESHOLD = 0.6

print(f"Cases with char_ratio_html_over_pdf < {LOW_RATIO_THRESHOLD}:")
low_ratio_df = jaccard_df[jaccard_df["char_ratio_html_over_pdf"] < LOW_RATIO_THRESHOLD].copy()
display(low_ratio_df.head(20))
print(f"Total flagged: {len(low_ratio_df):,}")

Cases with char_ratio_html_over_pdf < 0.6:


,doc_id,jaccard_word_unigram,html_chars,pymupdf_chars,char_ratio_html_over_pdf
1,2601.03262v1,0.456649,43052,82674,0.520744
2,2601.18886v1,0.467961,16749,30849,0.542935
4,2603.27055v1,0.476415,49473,97241,0.508767
7,2603.28589v1,0.495798,58415,99282,0.588375
10,2603.27269v1,0.514234,19385,37892,0.511586
15,2603.29846v1,0.544694,33732,60939,0.553537
21,2603.21440v3,0.574074,22263,42391,0.525182
29,2512.16236v1,0.594102,24281,47171,0.514744
33,2602.13543v1,0.598744,43781,84229,0.519785
38,2603.22966v1,0.602733,40310,73568,0.547928


Total flagged: 11


In [46]:

# Phân tích nguyên nhân Jaccard thấp
print("=== Phân tích nguyên nhân Jaccard thấp ===\n")

low_jac = jaccard_df[jaccard_df["jaccard_word_unigram"] < 0.45].copy()
low_jac["cause"] = "unknown"

# Case 1: HTML dài hơn PDF nhiều → math alt-text / captions thừa
low_jac.loc[low_jac["char_ratio_html_over_pdf"] > 1.3, "cause"] = "html_extra_content (math/captions)"
# Case 2: HTML ngắn hơn PDF nhiều → HTML bị hụt
low_jac.loc[low_jac["char_ratio_html_over_pdf"] < 0.5, "cause"] = "html_missing_content"
# Case 3: cân bằng nhưng Jaccard vẫn thấp → nội dung khác nhau thực sự
low_jac.loc[
    (low_jac["char_ratio_html_over_pdf"].between(0.5, 1.3)) & (low_jac["cause"] == "unknown"),
    "cause"
] = "content_divergence"

print("Phân bố nguyên nhân:")
print(low_jac["cause"].value_counts().to_string())
print(f"\nTổng case Jaccard < 0.45: {len(low_jac):,} / {len(jaccard_df):,} ({100*len(low_jac)/max(len(jaccard_df),1):.1f}%)")
display(low_jac[["doc_id", "jaccard_word_unigram", "html_chars", "pymupdf_chars", "char_ratio_html_over_pdf", "cause"]].head(20))


=== Phân tích nguyên nhân Jaccard thấp ===

Phân bố nguyên nhân:
cause
html_extra_content (math/captions)    1

Tổng case Jaccard < 0.45: 1 / 1,413 (0.1%)


,doc_id,jaccard_word_unigram,html_chars,pymupdf_chars,char_ratio_html_over_pdf,cause
0,2601.03474v2,0.363293,85492,48408,1.766072,html_extra_content (math/captions)


In [47]:
# ===== Export JSONL cho near-dup (đã lọc ratio ở bước trên) =====
# Không lọc thêm Jaccard — MinHashLSH so HTML-vs-HTML, Jaccard thấp vì alttext math
# không ảnh hưởng đến near-dup detection trên HTML corpus.

neardup_doc_ids = set(neardup_ids)
neardup_records = [r for r in records if r.get("doc_id") in neardup_doc_ids]

neardup_jsonl = PARSED_DIR / "arxiv_html_reference_v2_neardup.jsonl"
with open(neardup_jsonl, "w", encoding="utf-8") as f:
    for rec in neardup_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Near-dup JSONL: {neardup_jsonl}")
print(f"  Records: {len(neardup_records):,}  (dropped {len(records) - len(neardup_records):,} HTML hụt)")

# Alias cho cell export tiếp theo
keep_df        = jaccard_df      # jaccard_df đã là tập filtered
clean_records  = neardup_records
strict_records = neardup_records


Near-dup JSONL: arxiv_html_reference_workspace_v2\parsed_jsonl\arxiv_html_reference_v2_neardup.jsonl
  Records: 1,413  (dropped 166 HTML hụt)


In [48]:
def preview_doc(doc_id: str, head_chars: int = 1200):
    html_text = html_map.get(doc_id, "")
    pdf_text  = pymupdf_map.get(doc_id, "")

    print("=" * 80)
    print("doc_id:", doc_id)
    row = jaccard_df[jaccard_df["doc_id"] == doc_id]
    if len(row):
        print(row.iloc[0].to_dict())

    print("\n--- HTML preview ---\n")
    print(html_text[:head_chars])

    print("\n--- PyMuPDF preview ---\n")
    print(pdf_text[:head_chars])

# Ví dụ:
# preview_doc("2603.26221v1")

## 11. Xuất kết quả Jaccard

In [49]:
jaccard_csv  = ANALYSIS_DIR / "html_vs_pymupdf_jaccard.csv"
summary_json = ANALYSIS_DIR / "html_vs_pymupdf_jaccard_summary.json"

jaccard_df.to_csv(jaccard_csv, index=False, encoding="utf-8")

with open(summary_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Saved:")
print(f"  Jaccard CSV  : {jaccard_csv}  ({len(jaccard_df):,} rows, sau lọc ratio)")
print(f"  Summary JSON : {summary_json}")
print(f"  Near-dup JSONL: {PARSED_DIR / 'arxiv_html_reference_v2_neardup.jsonl'}  ({len(clean_records):,} records)")
print()
print(json.dumps(summary, indent=2, ensure_ascii=False))


Saved:
  Jaccard CSV  : arxiv_html_reference_workspace_v2\analysis\html_vs_pymupdf_jaccard.csv  (1,413 rows, sau lọc ratio)
  Summary JSON : arxiv_html_reference_workspace_v2\analysis\html_vs_pymupdf_jaccard_summary.json
  Near-dup JSONL: arxiv_html_reference_workspace_v2\parsed_jsonl\arxiv_html_reference_v2_neardup.jsonl  (1,413 records)

{
  "n_html_ok": 1418,
  "n_pymupdf_ok": 1578,
  "n_common_valid": 1418,
  "n_after_ratio_filter": 1413,
  "n_dropped_low_ratio": 5,
  "ratio_min": 0.5,
  "mean_jaccard": 0.8162956582011195,
  "median_jaccard": 0.8284671532846716,
  "p10_jaccard": 0.690597107236931,
  "p90_jaccard": 0.9324151067116401
}


## 12. Gợi ý dùng kết quả

### Nguyên nhân Jaccard thấp thường gặp

| Nguyên nhân | Biểu hiện | Xử lý |
|---|---|---|
| Math alttext trong HTML (LaTeX) | ratio > 1.3, Jaccard thấp | Heavy normalize đã xử lý |
| Citation key dạng `lewis2020retrieval` | Token thừa trong HTML | Heavy normalize lọc (chứa số) |
| Bảng/figure trong PDF không có trong HTML | ratio < 0.5, miss keyword thực nghiệm | **Loại khỏi benchmark** |
| HTML bị hụt toàn bộ section | ratio < 0.35 | **Loại khỏi benchmark** |

### Rule lọc thực dụng

```
fetch_status == "ok"                         # HTML fetch thành công
AND char_ratio_html_over_pdf >= 0.50         # HTML đủ nội dung so với PDF
AND jaccard_word_unigram (heavy) >= 0.55     # Nội dung match tốt sau normalize
```

- `ratio >= 0.50` và `Jaccard_heavy >= 0.55` → **giữ** (clean benchmark)
- `0.35 <= ratio < 0.50` → **flag** `warn_low_coverage_vs_pdf`, inspect thủ công
- `ratio < 0.35` → **loại**, HTML bị hụt quá nặng

### Lưu ý về `normalize_for_jaccard`
Hàm hiện tại (heavy) xóa:
- Token chứa `\`, `^`, `{`, `}` → LaTeX alttext
- Token chứa `_` ≥ 2 lần → citation key hoặc math symbol
- Số nguyên standalone
→ **Không nên dùng raw Jaccard hoặc light normalize** để đánh giá bài arXiv HTML vs PDF
